In [1]:
import os
import json
import time
import random
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

from openai import OpenAI
oai = OpenAI()

llm = ChatOpenAI(model="gpt-4o-mini")


/home/oncreative/anaconda3/envs/modu/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.3.0) or chardet (7.4.3)/charset_normalizer (3.4.7) doesn't match a supported version!
  warnings.warn(


In [2]:
WORK_DIR = Path('./260509_ft')
WORK_DIR.mkdir(exist_ok=True)

In [3]:
from dataclasses import dataclass, field

In [4]:
@dataclass
class FTState:
    train_path : Path | None = None
    val_path : Path | None = None
    train_field_id: str | None = None
    val_file_id: str | None = None
    job_id: str | None = None
    ft_model: str | None = None
    notes: list = field(default_factory=list)

In [5]:
state = FTState()

In [7]:
for msg in ['start', 'data ready', 'upload done']:
    state.notes.append(msg)

In [8]:
state

FTState(train_path=None, val_path=None, train_field_id=None, val_file_id=None, job_id=None, ft_model=None, notes=['start', 'data ready', 'upload done'])

In [ ]:
# 입력 : 무례하거나 짧은 한국어
# 출력 : 같은 의미의 정중한 한국어 이메일 문장

In [10]:
SYSTEM_MSG = "당신은 무례한 문장을 정중한 이메일 문장으로 바꿔주는 어시스턴트입니다."

TASK_SPEC = {
    'name' : 'polite_email_rewriter',
    'input' : '무례하거나 짧은 한국어 한 줄',
    'output' : '정중한 한국어 이메일 문장 1~2 문장',
    'min_examples' : 30,
    'system' : SYSTEM_MSG
}

print(json.dumps(TASK_SPEC, ensure_ascii=False, indent=2))

{
  "name": "polite_email_rewriter",
  "input": "무례하거나 짧은 한국어 한 줄",
  "output": "정중한 한국어 이메일 문장 1~2 문장",
  "min_examples": 30,
  "system": "당신은 무례한 문장을 정중한 이메일 문장으로 바꿔주는 어시스턴트입니다."
}


In [11]:
seed_pairs = [
    ("회의 내일로 미뤄.",   "회의를 내일로 변경 가능할지 여쭙고 싶습니다."),
    ("문서 빨리 줘.",       "문서 전달 가능 시점을 알려주실 수 있을까요?"),
    ("이거 다시 해.",       "이 부분은 한 번 더 수정 부탁드려도 될까요?"),
    ("일정 잡아.",          "편하신 일정을 알려주시면 회의를 잡아 두겠습니다."),
    ("왜 답장 안 해.",      "혹시 이전 메일을 확인하셨을지 여쭙고 싶습니다."),
    ("이건 틀렸어.",        "이 부분은 다시 한 번 검토가 필요해 보입니다."),
]

In [12]:
def to_jsonl_line(system, user, assistant) -> str:
    return json.dumps({'messages' : [
        {'role' : 'system', 'content' : system},
        {'role' : 'user', 'content' : user},
        {'role' : 'assistant', 'content' : assistant},
    ]}, ensure_ascii=False)

In [13]:
seed_path = WORK_DIR / 'seed.jsonl'  # Path('./260509_ft/seed.jsonl')
with open(seed_path, 'w', encoding='utf-8') as f:
    for r, p in seed_pairs:
        f.write(to_jsonl_line(SYSTEM_MSG, r, p) + '\n')

In [14]:
extras = [
    ("자료 빠뜨렸어.",     "자료 한 가지가 누락된 것 같습니다. 확인 부탁드립니다."),
    ("일정 다시 봐.",      "일정을 다시 한 번 확인해 주실 수 있을까요?"),
    ("이름 잘못 적었어.",  "이름 표기를 한 번 더 확인해 주시면 감사하겠습니다."),
    ("결재 빨리 해.",      "결재 처리 가능 시점을 알려주실 수 있을까요?"),
]

with open(seed_path, 'a', encoding='utf-8') as f:
    for r, p in seed_pairs:
        f.write(to_jsonl_line(SYSTEM_MSG, r, p) + '\n')

In [15]:
import re
AUG_SYS = '당신은 학습용 데이터 페어를 생성하는 전문가입니다. 출력은 반드시 JSON 배열로'
def make_augment_prompt(pairs, n=8):
    sample = '\n'.join(f'- "{r}" -> "{p}"' for r, p in pairs[:4])
    return (f"다음은 무례한 문장 -> 정중한 이메일 문장 페어입니다.\n"
            f"같은 스타일,길이 분포로 새 페어 {n} 쌍을 한국어로 만드세요.\n"
            f"출력 형식: [{{\"rude\":\"...\",\"polite\":\"...\"}}, ...]\n\n예시:\n{sample}")

resp = llm.invoke([SystemMessage(content=AUG_SYS), 
                  HumanMessage(content=make_augment_prompt(seed_pairs, n=10))]).content
print(resp[:100])

[
    {"rude":"이거 왜 이렇게 해?", "polite":"이 부분에 대해 다시 설명해 주실 수 있을까요?"},
    {"rude":"답장 왜 이렇게 늦어?", "po


In [17]:
print(resp[:300])

[
    {"rude":"이거 왜 이렇게 해?", "polite":"이 부분에 대해 다시 설명해 주실 수 있을까요?"},
    {"rude":"답장 왜 이렇게 늦어?", "polite":"답변이 늦어진 점에 대해 양해 부탁드립니다."},
    {"rude":"자료 더 없어?", "polite":"자료가 추가로 있으신지 여쭤봐도 될까요?"},
    {"rude":"지금 회의 해.", "polite":"지금 회의 진행이 가능하신지 문의드립니다."},
    {"rude":"제 시간에 와.", "polite":"정해진 시간에 참석


In [ ]:
# gpt-4o-mini -> gpt-4o-mini-260519 -> gpt-4o-mini-260519_2

In [18]:
def parse_pairs(text):
    m = re.search(r"\[.*\]", text, re.DOTALL)
    if not m:
        return []
    try:
        return [(it['rude'], it['polite']) for it in json.loads(m.group(0))]
    except Exception:
        return []

augmented = parse_pairs(resp)

In [19]:
augmented

[('이거 왜 이렇게 해?', '이 부분에 대해 다시 설명해 주실 수 있을까요?'),
 ('답장 왜 이렇게 늦어?', '답변이 늦어진 점에 대해 양해 부탁드립니다.'),
 ('자료 더 없어?', '자료가 추가로 있으신지 여쭤봐도 될까요?'),
 ('지금 회의 해.', '지금 회의 진행이 가능하신지 문의드립니다.'),
 ('제 시간에 와.', '정해진 시간에 참석해 주시면 감사하겠습니다.'),
 ('그거 너가 해.', '이 부분에 대해서는 귀하께서 맡아 주실 수 있을까요?'),
 ('이메일 확인해.', '이메일 확인해 주시면 감사하겠습니다.'),
 ('이거 빨리 끝내.', '이 일을 가능한 한 빨리 처리해 주실 수 있을까요?'),
 ('다시 보내줘.', '다시 보내주실 수 있으신지 부탁드립니다.'),
 ('늦지 말고 와.', '정시에 참석해 주시면 대단히 감사하겠습니다.')]

In [ ]:
# -> llm -> evaluate -> 좋은애들만 남겨라

In [20]:
import tiktoken

In [ ]:
# 구조, 길이(빈 응답), 중복 

In [21]:
enc = tiktoken.get_encoding('cl100k_base')

In [22]:
MAX_TOKENS_PER_SAMPLE = 4096

In [23]:
def check_sample(row) -> list[str]:
    errs = []
    msgs = row.get('messages', [])
    roles = {m.get('role') for m in msgs}
    if not {'system', 'user', 'assistant'} <= roles:
        errs.append('필수 role 누락')
    if not any(m.get('role') == 'assistant' and m.get('content') for m in msgs):
        errs.append('assistant 응답 누락')
    total = sum(len(enc.encode(m.get('content', ''))) for m in msgs)
    if total > MAX_TOKENS_PER_SAMPLE:
        errs.append('토큰 개수 초과')
    return errs

In [24]:
def validate(path):
    seen, bad, rows = set(), [], []
    for i, line in enumerate(open(path, encoding='utf-8')):
        try:
            row = json.loads(line)
        except:
            bad.append((i, "json 파싱 실패"))
            continue
        errs = check_sample(row)
        user_msg = next((m['content'] for m in row.get('messages', []) if m['role'] == 'user'), "")
        if user_msg in seen:
            errs.append('user 중복')
        seen.add(user_msg)
        (bad if errs else rows).append((i, ", ".join(errs)) if errs else row)
    return rows, bad

In [25]:
seed_path

PosixPath('260509_ft/seed.jsonl')

In [30]:
# 페어 리스트 -> jsonl 빌드
def pairs_to_jsonl(pairs : list[tuple[str, str]], system : str, out_path : Path) -> int:
    n=0
    with open(out_path, 'w', encoding='utf-8') as f:
        for r, p in pairs:
            f.write(to_jsonl_line(system, r, p) + "\n")
            n +=1
    return n

In [31]:
augmented

[('이거 왜 이렇게 해?', '이 부분에 대해 다시 설명해 주실 수 있을까요?'),
 ('답장 왜 이렇게 늦어?', '답변이 늦어진 점에 대해 양해 부탁드립니다.'),
 ('자료 더 없어?', '자료가 추가로 있으신지 여쭤봐도 될까요?'),
 ('지금 회의 해.', '지금 회의 진행이 가능하신지 문의드립니다.'),
 ('제 시간에 와.', '정해진 시간에 참석해 주시면 감사하겠습니다.'),
 ('그거 너가 해.', '이 부분에 대해서는 귀하께서 맡아 주실 수 있을까요?'),
 ('이메일 확인해.', '이메일 확인해 주시면 감사하겠습니다.'),
 ('이거 빨리 끝내.', '이 일을 가능한 한 빨리 처리해 주실 수 있을까요?'),
 ('다시 보내줘.', '다시 보내주실 수 있으신지 부탁드립니다.'),
 ('늦지 말고 와.', '정시에 참석해 주시면 대단히 감사하겠습니다.')]

In [32]:
aug_path = WORK_DIR / 'aug.jsonl'
n = pairs_to_jsonl(augmented, SYSTEM_MSG, aug_path)
n

10

In [33]:
combined = WORK_DIR / 'combined.jsonl'
with open(combined, 'w', encoding='utf-8') as f:
    for src in [seed_path, aug_path]:
        if src.exists():
            f.write(open(src).read())

ok, bad = validate(combined)

In [34]:
len(ok), len(bad)

(16, 6)

In [35]:
bad

[(6, 'user 중복'),
 (7, 'user 중복'),
 (8, 'user 중복'),
 (9, 'user 중복'),
 (10, 'user 중복'),
 (11, 'user 중복')]

In [36]:
def filter_clean(in_path:Path, out_path:Path) -> dict:  # jsonl 파일을 받아서 -> validate Ok, Bad -> ok만 out_path에 저장
    ok_rows, bad_rows = validate(in_path)
    with open(out_path, 'w', encoding='utf-8') as f:
        for row in ok_rows:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')
    total = len(ok_rows) + len(bad_rows)
    pct = (len(ok_rows) / total * 100) if total else 0.0
    return {'ok' : len(ok_rows), 'bad' : len(bad_rows), 'pass_rate' : pct}

In [37]:
clean_path = WORK_DIR / 'clean.jsonl'
res = filter_clean(combined, clean_path)

In [38]:
res

{'ok': 16, 'bad': 6, 'pass_rate': 72.72727272727273}

In [ ]:
# train /  test